# Local RAG - ChromaDB, LangChain, Ollama and Streamlit 

### Author: ***Naresh Padiyar***

This project implements a complete **local RAG pipeline** that allows an LLM to retrieve relevant information from private data sources before generating a response.

## 📂 Supported Private Datasources

The project supports a variety of locally stored data sources.

| Datasource | File Format |
|---|---|
| PDF | `.pdf` |
| Text | `.txt` |
| CSV | `.csv` |
| Excel | `.xls`, `.xlsx` |
| SQL | `.sql` / SQL Databases |
| Word | `.doc`, `.docx` |
| PowerPoint | `.ppt`, `.pptx` |
| HTML | `.html`, `.htm` |
| JSON | `.json` |
| Markdown | `.md` |

## Architecture Diagram

![RAG-Explaination.png](architecture/RAG-Architecture.png)

#### A **Local RAG** (Retrieval-Augmented Generation) is a RAG system that runs entirely on your local hardware - such as your personal laptop or an on-premise server-without relying on cloud APIs like OpenAI, Anthropic, or Google.

---

## Core Components

A local RAG setup typically replaces cloud services with local equivalents across its three main phases:

* **Local Embedding Model:** Converts your documents into vector representations using lightweight local models (e.g., via Hugging Face, Ollama, or sentence-transformers).
* **Local Vector Database:** Stores and indexes your document embeddings locally (e.g., **Chroma**, **FAISS**, **LanceDB**, or **Qdrant** embedded mode).
* **Local LLM:** Generates the final response using an open-weights model running locally via engines like **Ollama**, **LM Studio**, **llama.cpp**, or **vLLM**.

Here is the role of each component in a local RAG stack:

* **ChromaDB (The Vector Database):** Stores your document embeddings locally and performs fast semantic searches to find the text chunks most relevant to your query.
* **LangChain (The Orchestration Framework):** Acts as the "glue" that connects your data sources, vector store, and LLM, managing the end-to-end pipeline (document loading, chunking, prompt building).
* **Ollama (The Local LLM Runner):** Runs open-source language models (like Llama 3 or Mistral) directly on your local hardware to generate the final answers.
* **Streamlit (The User Interface):** Provides a simple, Python-based web app interface so you can easily upload documents, view chat history, and interact with your local RAG system.

---

### Key Advantages

* **Data Privacy & Security:** Your sensitive files, documents, and queries never leave your local machine, making it ideal for proprietary code, personal journals, or confidential enterprise data.
* **Zero API Costs:** Once you have the hardware, there are no token-based usage fees or recurring subscription costs.
* **Offline Capability:** Works entirely without an internet connection once the models and tools are downloaded.
* **Full Customization:** You have complete control over prompt templates, chunking strategies, embedding models, and LLM parameters.

---

### Trade-offs & Challenges

* **Hardware Requirements:** Running both an embedding model, vector store, and a local LLM requires decent hardware—specifically a modern CPU, plenty of RAM, and ideally a dedicated GPU with sufficient VRAM (e.g., an NVIDIA card with 8GB+ VRAM or a unified-memory Apple Silicon Mac).
* **Setup Complexity:** Unlike managed cloud APIs, you have to configure and wire together the orchestration framework (like LangChain or LlamaIndex), vector store, and local runtime yourself.
* **Performance vs. Cloud:** Local models (like Llama 3 or Mistral in smaller parameter sizes) generally have lower reasoning capabilities and slower generation speeds compared to massive frontier cloud models.

## Prerequisites

Before running the Local RAG application, make sure the following dependencies and tools are installed and configured.

### 1. Install the Required Python Dependencies

Install the required Python libraries using `pip`:

```bash
pip3 install \
    chromadb \
    langchain \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    langchain-text-splitters \
    streamlit \
    ollama \
    pypdf \
    docx2txt \
    unstructured \
    "unstructured[all-docs]" \
    beautifulsoup4 \
    openpyxl
```

> **Recommendation:** It is recommended to use a Python virtual environment to keep the project dependencies isolated.

Example:

```bash
python3 -m venv .venv
source .venv/bin/activate
```

On Windows:

```powershell
.venv\Scripts\activate
```

---

### 2. Install Ollama

Ollama is used to run the **LLM and embedding models locally**.

Download and install Ollama from the official website:

[https://ollama.com/download](https://ollama.com/download)

After installation, verify that Ollama is available:

```bash
ollama --version
```

---

### 3. Download the Embedding Model

This project uses **`mxbai-embed-large`** to generate vector embeddings for the documents.

Pull the model using:

```bash
ollama pull mxbai-embed-large
```

Verify that the model has been downloaded:

```bash
ollama list
```

You should see something similar to:

```text
NAME                    SIZE
mxbai-embed-large       ...
```

---

### 4. Download a Local LLM

The RAG application also requires a local LLM to generate responses.

For example, to use **Llama 3.2**:

```bash
ollama pull llama3.2
```

Verify the installed models:

```bash
ollama list
```

You should see both the embedding model and LLM:

```text
NAME
mxbai-embed-large
llama3.2
```

---

### Prerequisites Summary

Before starting the application, ensure you have:

- ✅ Python 3.x
- ✅ Required Python packages installed
- ✅ Ollama installed and running
- ✅ `mxbai-embed-large` embedding model downloaded
- ✅ A supported local LLM such as `llama3.2` downloaded
- ✅ Your private datasource files available under the `datasource/` directory

In [1]:
# Data Ingestion - Private documents are loaded using the appropriate LangChain document loader.
from pathlib import Path

from langchain_community.document_loaders import (
    CSVLoader,
    PyPDFLoader,
    TextLoader,
    BSHTMLLoader,
    Docx2txtLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredPowerPointLoader,
    UnstructuredExcelLoader,
    UnstructuredMarkdownLoader
)


def CSVLoaderFunc(files):
    docs = []
    for file in files:
        docs.extend(CSVLoader(file).load())
    return docs


def DOCLoader(files):
    docs = []
    for file in files:
        # Legacy .doc files need the unstructured loader.
        if Path(file).suffix.lower() == ".doc":
            loader = UnstructuredWordDocumentLoader(file)
        else:
            loader = Docx2txtLoader(file)
        docs.extend(loader.load())
    return docs


def HTMLLoader(files):
    docs = []
    for file in files:
        docs.extend(BSHTMLLoader(file).load())
    return docs


def JSONLoaderFunc(files):
    docs = []
    for file in files:
        # Preserve the original JSON content as text.
        docs.extend(TextLoader(file, encoding="utf-8").load())
    return docs


def MDLoader(files):
    docs = []
    for file in files:
        docs.extend(UnstructuredMarkdownLoader(file).load())
    return docs


def PDFLoader(files):
    docs = []
    for file in files:
        docs.extend(PyPDFLoader(file).load())
    return docs


def PPTLoader(files):
    docs = []
    for file in files:
        docs.extend(UnstructuredPowerPointLoader(file).load())
    return docs


def SQLLoader(files):
    docs = []
    for file in files:
        # SQL files are loaded as text, including queries and schemas.
        docs.extend(TextLoader(file, encoding="utf-8").load())
    return docs


def TXTLoader(files):
    docs = []
    for file in files:
        docs.extend(TextLoader(file, encoding="utf-8").load())
    return docs


def XLSLoader(files):
    docs = []
    for file in files:
        docs.extend(UnstructuredExcelLoader(file).load())
    return docs


/var/folders/lk/9n5s5gc55g71ty0h9s654xdr0000gn/T/ipykernel_55623/3590654537.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [2]:
# Iterate through each private data source category and use list comprehension to store its corresponding documents.
csv_files  = [str(f) for f in Path("datasource/CSV").rglob("*.csv")]

doc_files  = [
    str(f) for f in Path("datasource/DOC").rglob("*")
    if f.is_file() and f.suffix.lower() in [".doc", ".docx"]
]

html_files = [
    str(f) for f in Path("datasource/HTML").rglob("*")
    if f.is_file() and f.suffix.lower() in [".html", ".htm"]
]

json_files = [str(f) for f in Path("datasource/JSON").rglob("*.json")]

md_files   = [str(f) for f in Path("datasource/MD").rglob("*.md")]

pdf_files  = [str(f) for f in Path("datasource/PDF").rglob("*.pdf")]

ppt_files = [
    str(f) for f in Path("datasource/PPT").rglob("*")
    if f.is_file() and f.suffix.lower() in [".ppt", ".pptx"]
]

sql_files = [str(f) for f in Path("datasource/SQL").rglob("*.sql")]

txt_files = [str(f) for f in Path("datasource/TXT").rglob("*.txt")]

xls_files = [
    str(f) for f in Path("datasource/XLS").rglob("*")
    if f.is_file() and f.suffix.lower() in [".xls", ".xlsx"]
]

In [3]:
# Load documents
csv_docs = CSVLoaderFunc(csv_files)
doc_docs = DOCLoader(doc_files)
html_docs = HTMLLoader(html_files)
json_docs = JSONLoaderFunc(json_files)
md_docs = MDLoader(md_files)
pdf_docs = PDFLoader(pdf_files)
ppt_docs = PPTLoader(ppt_files)
sql_docs = SQLLoader(sql_files)
txt_docs = TXTLoader(txt_files)
xls_docs = XLSLoader(xls_files)

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1


In [4]:
# Combine all loaded documents for downstream processing
all_docs = (csv_docs + doc_docs + html_docs + json_docs + md_docs + pdf_docs + ppt_docs + sql_docs + txt_docs + xls_docs)
print(f"Total documents loaded: {len(all_docs)}")

Total documents loaded: 13


In [5]:
# Text Splitting - Large documents are divided into smaller chunks to improve retrieval accuracy.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

split_documents = text_splitter.split_documents(all_docs)

print(f"Total documents: {len(all_docs)}")
print(f"Total chunks: {len(split_documents)}")

Total documents: 13
Total chunks: 19


In [6]:
# Embeddings - Each document chunk is converted into a numerical vector using an Ollama embedding model. The generated embeddings are stored in ChromaDB.
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# Initialize Ollama Embeddings
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large"
)

# Set directory for persistent storage
persist_directory = "./chroma_db"

# Create and persist ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=split_documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

print("✅ Data successfully stored in ChromaDB!")

# Reload the vector store for retrieval
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)

print("🔄 ChromaDB reloaded successfully!")

✅ Data successfully stored in ChromaDB!
🔄 ChromaDB reloaded successfully!


In [ ]:
# Retrieval - The most relevant document chunks are retrieved using semantic similarity search.
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# 1. Initialize LLM
# ============================================================

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)


# ============================================================
# 2. Create Retriever
# ============================================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)


# ============================================================
# 3. Create Prompt
# ============================================================

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant.

Answer the question using only the context provided below.

If the answer cannot be found in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:"""
)


# ============================================================
# 4. User Query
# ============================================================

query = "What is hotel name?"


# ============================================================
# 5. Retrieve Relevant Documents
# ============================================================

documents = retriever.invoke(query)

print(f"Retrieved {len(documents)} documents")


# ============================================================
# 6. Combine Retrieved Documents
# ============================================================

context = "\n\n".join(
    doc.page_content
    for doc in documents
)


# ============================================================
# 7. Create Prompt
# ============================================================

messages = prompt.invoke({
    "context": context,
    "question": query
})


# ============================================================
# 8. Generate Answer
# ============================================================

response = llm.invoke(messages)


# ============================================================
# 9. Display Answer
# ============================================================

print("\nAnswer:")
print(response.content)


# ============================================================
# 10. Display Sources
# ============================================================

print("\nSources:")

for i, doc in enumerate(documents, 1):
    print(f"\n--- Source {i} ---")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))

### Start the Streamlit Application

Open a terminal or shell command prompt and run the following command to start the interactive Streamlit application. The application will open in a browser window.

```
streamlit run app.py
```

### Below is the streamlit app code for reference

In [ ]:
'''

import streamlit as st

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# Streamlit Configuration
# ============================================================

st.set_page_config(layout="wide")
st.title("My Local Chatbot")


# ============================================================
# Sidebar Settings
# ============================================================

st.sidebar.header("Settings")

MODEL = st.sidebar.selectbox(
    "Choose a Model",
    ["llama3.2", "deepseek-r1:1.5b"],
    index=0
)

MAX_HISTORY = st.sidebar.number_input(
    "Max History",
    min_value=1,
    max_value=10,
    value=2
)

CONTEXT_SIZE = st.sidebar.number_input(
    "Context Size",
    min_value=1024,
    max_value=16384,
    value=8192,
    step=1024
)


# ============================================================
# Session State
# ============================================================

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if (
    "prev_context_size" not in st.session_state
    or st.session_state.prev_context_size != CONTEXT_SIZE
):
    st.session_state.prev_context_size = CONTEXT_SIZE


# ============================================================
# Initialize Ollama LLM
# ============================================================

llm = ChatOllama(
    model=MODEL,
    temperature=0,
    num_ctx=CONTEXT_SIZE,
    streaming=True
)


# ============================================================
# Initialize Ollama Embeddings
# ============================================================

embeddings = OllamaEmbeddings(
    model="mxbai-embed-large"
)


# ============================================================
# Initialize ChromaDB
# ============================================================

persist_directory = "./chroma_db"

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)


# ============================================================
# Create Retriever
# ============================================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)


# ============================================================
# RAG Prompt
# ============================================================

prompt_template = ChatPromptTemplate.from_template(
    """You are a helpful assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:"""
)


# ============================================================
# Display Chat History
# ============================================================

for msg in st.session_state.chat_history:

    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])


# ============================================================
# Trim Chat History
# ============================================================

def trim_history():

    max_messages = MAX_HISTORY * 2

    if len(st.session_state.chat_history) > max_messages:

        st.session_state.chat_history = (
            st.session_state.chat_history[-max_messages:]
        )


# ============================================================
# Handle User Input
# ============================================================

if user_prompt := st.chat_input("Say something"):

    # --------------------------------------------------------
    # Add user message to history
    # --------------------------------------------------------

    st.session_state.chat_history.append(
        {
            "role": "user",
            "content": user_prompt
        }
    )

    with st.chat_message("user"):
        st.markdown(user_prompt)

    trim_history()


    # --------------------------------------------------------
    # Generate response
    # --------------------------------------------------------

    with st.chat_message("assistant"):

        response_container = st.empty()

        # Retrieve relevant documents
        retrieved_docs = retriever.invoke(user_prompt)

        if not retrieved_docs:

            full_response = "No relevant documents found."

        else:

            # ------------------------------------------------
            # Combine retrieved documents into context
            # ------------------------------------------------

            context = "\n\n".join(
                doc.page_content
                for doc in retrieved_docs
            )

            # ------------------------------------------------
            # Create prompt
            # ------------------------------------------------

            messages = prompt_template.invoke(
                {
                    "context": context,
                    "question": user_prompt
                }
            )

            # ------------------------------------------------
            # Invoke LLM
            # ------------------------------------------------

            response = llm.invoke(messages)

            full_response = response.content

        # ----------------------------------------------------
        # Display response
        # ----------------------------------------------------

        response_container.markdown(full_response)

    # --------------------------------------------------------
    # Save assistant response
    # --------------------------------------------------------

    st.session_state.chat_history.append(
        {
            "role": "assistant",
            "content": full_response
        }
    )

    trim_history()

'''